In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, random_split
import torchvision.datasets as datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
from matplotlib.ticker import ScalarFormatter
import humanize, math

class CLSViz:
  def __init__(self, base_img, patch_size):
    self.fig, self.ax = plt.subplots()
    self.ax.axis("off")

    img = base_img.permute(1, 2, 0) if base_img.shape[-1] != 3 else base_img
    g = img.shape[0] // patch_size
    heatmap = torch.zeros((g, g)).numpy()

    self.img_artist = self.ax.imshow(img)
    self.heatmap_artist = self.ax.imshow(heatmap, cmap="magma", alpha=0.45, vmin=0, vmax=1)

    plt.ioff()
    plt.show(block=False)

  def update(self, block_cls_scores):
    # Average each set of normalized attention scores corresponding to
    # the [cls] token across all layers: [L, B, N * N] -> [B, N * N]
    heatmap = block_cls_scores.detach().cpu().mean(dim=0)
    # This is currently only visualizing one image at once
    assert heatmap.shape[0] == 1
    heatmap = heatmap[0]
    # Overlay the [N, N] heatmap over the [H, W] image
    heatmap = heatmap.reshape(math.sqrt(heatmap.shape[-1]))
    scale = self.img_artist.get_array().shape[0] // heatmap.shape[-1]
    heatmap = heatmap.repeat_interleave(scale, dim=-2)
    heatmap = heatmap.repeat_interleave(scale, dim=-1)

    self.heatmap_artist.set_data(heatmap.numpy())
    self.fig.canvas.draw_idle()
    self.fig.canvas.flush_events()
    plt.pause(0.001)

def plot_loss_curves(train_losses, val_losses, output_path):
  train_losses = torch.tensor(train_losses)
  val_losses = torch.tensor(val_losses)

  fig, ax = plt.subplots()
  formatter = ScalarFormatter()
  formatter.set_scientific(False)

  ax.set_title("Loss Curves")
  ax.set_xlabel("Epochs")
  ax.set_ylabel("Loss")
  ax.set_yscale("log")
  ax.yaxis.set_major_formatter(formatter)

  ax.plot(train_losses, "b-")
  ax.plot(val_losses, "o-")

  stats = f"min: {train_losses.min().item()}, max: {train_losses.max().item()}, " + \
          f"mean: {train_losses.mean().item()}, std: {train_losses.std().item()}, "
  ax.text(0.5, -0.18, f"Train loss | {stats}",
          transform=ax.transAxes, ha="center", va="top", fontsize=10)

  stats = f"min: {val_losses.min().item()}, max: {val_losses.max().item()}, " + \
          f"mean: {val_losses.mean().item()}, std: {val_losses.std().item()}, "
  ax.text(0.5, -0.36, f"Validation loss | {stats}",
          transform=ax.transAxes, ha="center", va="top", fontsize=10)

  fig.tight_layout()
  fig.savefig(output_path, bbox_inches="tight")
  plt.show()


In [ ]:
def custom_lr_scheduler(step, warmup, total):
  if step < warmup:
    return float(step) / float(max(1, warmup))

  # Cosine decay phase
  progress = float(step - warmup) / float(max(1, total - warmup))
  progress = min(1.0, max(0.0, progress))
  return 0.5 * (1.0 + math.cos(math.pi * progress))

def patchify_img(img_batch, B, C, P, G):
  # Convert image into a sequence of patches
  patches = img_batch.unfold(2, P, P).unfold(3, P, P)
  patches = patches.contiguous().view(B, C, G * G, P, P)
  patches = patches.permute(0, 2, 3, 4, 1)
  patches = patches.flatten(start_dim=2, end_dim=4)
  return patches

def init_weights(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    if module.bias is not None:
      nn.init.zeros_(module.bias)

class RMSNorm(nn.Module):
  def __init__(self, dim):
    self.scale = nn.Parameter(torch.zeros(dim))

  def forward(self, x):
    rms = torch.sqrt((x ** 2).sum(dim=-1) / x.shape[-1])
    return x * self.scale / rms

class Block(nn.Module):
  def __init__(self, embed_dim, num_heads):
    super().__init__()
    self.norm1 = nn.RMSNorm(embed_dim)
    self.msa = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
    self.scale1 = nn.Parameter(torch.zeros(embed_dim))
    self.norm2 = nn.RMSNorm(embed_dim)
    self.ffn = nn.Sequential(
      nn.Linear(embed_dim, embed_dim * 4),
      nn.GELU(),
      nn.Linear(embed_dim * 4, embed_dim))
    self.scale2 = nn.Parameter(torch.zeros(embed_dim))

  def forward(self, prev_output):
    n = self.norm1(prev_output)
    z1, attn_scores = self.msa(n, n, n)
    cls_scores = attn_scores[:, 0]
    z1 = z1 @ self.scale1
    z1 = z1 + prev_output
    output = self.ffn(self.norm2(z1)) + z1
    return output, cls_scores

class ViT(nn.Module):
  def __init__(self, patch_size, grid_size, embed_dim,
               layers, num_heads, num_classes):
    super().__init__()
    size = 3 * patch_size * patch_size
    elements = grid_size * grid_size + 1

    self.embed_filters = nn.Linear(size, embed_dim, bias=False)
    self.class_token = nn.Parameter(torch.zeros(embed_dim))
    self.pos_embed = nn.Parameter(torch.zeros(elements, embed_dim))
    nn.init.normal_(self.class_token, std=0.02)
    nn.init.normal_(self.pos_embed, std=0.02)

    self.blocks = nn.ModuleList([Block(embed_dim, num_heads) for _ in range(layers)])
    self.class_norm = nn.LayerNorm(embed_dim)
    self.class_proj = nn.Linear(embed_dim, num_classes)

  # P = patch size, N = patch count, C = channel count
  # B = batch size, O = number of output classes
  # Input shape: (B, N, P * P * C), output shape: (B, O)
  def forward(self, x, store_cls_scores):
    embeddings = self.embed_filters(x)
    tok = self.class_token.unsqueeze(0).unsqueeze(0)
    tok = tok.expand(x.shape[0], -1, -1)
    tokens = torch.cat((tok, embeddings), dim=1)
    state = tokens + self.pos_embed

    block_cls_scores = []
    for block in self.blocks:
      state, cls_scores = block(state)
      if store_cls_scores:
        # TODO: wait, ... this isn't nxn, it's nx1, how are we supposed to overlay this?
        block_cls_scores.append(cls_scores[:, 1:]) # Drop the [cls] key
    scores = torch.stack(block_cls_scores) if store_cls_scores else None

    class_token = self.class_norm(state[:, 0])
    output = self.class_proj(class_token)
    return output, scores

class Harness:
  def __init__(self, hyperparams, img_size, device):
    self.h = hyperparams
    self.patches_per_side = img_size // self.h["patch_size"]
    self.device = device

    self.model = ViT(
      self.h["patch_size"],
      self.patches_per_side,
      self.h["embedding_dim"],
      self.h["layers"],
      self.h["num_heads"],
      self.h["num_classes"],
    ).apply(init_weights).to(self.device)
    self.num_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)

    self.optimizer = AdamW(self.model.parameters(),
                           lr=self.h["learning_rate"],
                           betas=self.h["betas"])
    self.criterion = nn.CrossEntropyLoss()

  def run_model(self, images, labels, scheduler=None):
    # Images shape: (B, C, H, W), Patches shape: (B, N, P, P, C)
    images, labels = images.to(self.device), labels.to(self.device)
    patches = patchify_img(
      images, images.shape[0],
      images.shape[1], self.h["patch_size"],
      self.patches_per_side
    )

    prediction, scores = self.model(patches, scheduler is None)
    loss = self.criterion(prediction, labels)
    if scheduler is not None:
      loss.backward()
      self.optimizer.step()
      self.optimizer.zero_grad(set_to_none=False)
      scheduler.step()

    return scores

In [ ]:
torch.manual_seed(67)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hp = {
  "epochs": 25,
  "warmup_epochs": 3,
  "batch_size": 64,
  "learning_rate": 1e-3,
  "betas": [0.9, 0.999],
  "embedding_dim": 256,
  "patch_size": 7,
  "num_heads": 8,
  "layers": 12,
  "num_classes": 10,
}

p = v2.Compose([ v2.RGB(), v2.ToImage(), v2.ToDtype(torch.float32, scale=True) ])
dataset = datasets.MNIST(root=".cache/mnist", train=True, download=True, transform=p)
train_set, val_set = random_split(dataset, [0.8, 0.2])
train_loader = DataLoader(train_set, shuffle=True, batch_size=hp["batch_size"])

rand_idx = torch.randint(0, len(train_set), size=(1,))
viz_img, viz_label = val_set[0][rand_idx], val_set[1][rand_idx]
viz_img, viz_label = viz_img.unsqueeze(0), viz_label.unsqueeze(0)

img_size = viz_img.shape[-1]
viz = CLSViz(viz_img[0], hp["patch_size"])
writer = FFMpegWriter(fps=24)

harness = Harness(hp, img_size, device)
ts = hp["epochs"] * len(train_loader)
ws = hp["warmup_epochs"] * len(train_loader)
scheduler = LambdaLR(harness.optimizer, lambda step: custom_lr_scheduler(step, ws, ts))

print(f"Training a {humanize.metric(harness.num_params)} model using {device}")
train_losses, val_losses = [], []

# Training loop
for epoch in range(hp["epochs"]):
  print("Training...")
  harness.model.train()

  with torch.enable_grad():
    for batch, (images, labels) in enumerate(train_loader):
      _ = harness.run_model(images, labels, scheduler=scheduler)

  print("Validating...")
  with torch.no_grad():
    _, block_cls_scores = harness.run_model(viz_img, viz_label)
    viz.update(block_cls_scores)